# ROGII Wellbore Geology - Dense (MLP) Training

Train a Multi-Layer Perceptron (Dense) model to predict **TVT** from horizontal well logs.

**Features (12):** `MD, X, Y, Z, ANCC, ASTNU, ASTNL, EGFDU, EGFDL, BUDA, GR, TVT_input`

**Label:** `TVT`

**Runtime**: Kaggle GPU (T4/P100) - Keras 3 + JAX backend

**Author**: Samir Attrah

### 1. Environment & Imports
Setup JAX backend for Keras and import necessary libraries.

In [1]:
# Cell 1: Environment & Imports
import os
os.environ["KERAS_BACKEND"] = "jax"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"] = "platform"

import jax
# Enable JAX float64 precision natively
jax.config.update("jax_enable_x64", True)

import keras
from keras import layers, callbacks, regularizers
import jax.numpy as jnp
import numpy as np
import polars as pl
import glob, pickle, warnings, random
warnings.filterwarnings("ignore")

# Show enough digits to round-trip float64 diagnostics
np.set_printoptions(precision=17, floatmode="unique")

# Set seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
keras.utils.set_random_seed(SEED)

print(f"Keras version : {keras.__version__}")
print(f"Keras backend : {keras.backend.backend()}")
print(f"Working dir   : {os.getcwd()}")


2026-08-01 01:31:55.106841: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1785537115.122485  198308 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1785537115.127157  198308 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Keras version : 3.12.0
Keras backend : jax
Working dir   : /home/samer/Documents/competitions/ROGII/notebooks


### 2. Auto-detect Dataset Path
Locate the competition dataset in local or Kaggle environments.

In [2]:
# Cell 2: Auto-detect dataset path

def find_data_dir():
    """Searches common Kaggle mount points for the ROGII dataset."""
    candidates = [
        "/kaggle/input/competitions/rogii-wellbore-geology-prediction",
        "/kaggle/input/rogii-wellbore-geology-prediction",
        "/home/samer/Documents/competitions/ROGII/dataset",
    ]

    for scan_root in ["/kaggle/input", "/kaggle/input/competitions"]:
        if os.path.isdir(scan_root):
            for entry in os.listdir(scan_root):
                full = os.path.join(scan_root, entry)
                if os.path.isdir(full) and full not in candidates:
                    candidates.append(full)

    print("Searching for ROGII dataset...")
    for path in candidates:
        if not os.path.isdir(path):
            continue

        contents = os.listdir(path)
        has_train = "train" in contents and os.path.isdir(os.path.join(path, "train"))
        n_train = 0
        if has_train:
            n_train = len(glob.glob(os.path.join(path, "train", "*__horizontal_well.csv")))

        if n_train > 0:
            print(f"  V Using {path} (found {n_train} train wells)")
            return path

    raise FileNotFoundError("Could not find ROGII dataset.")

DATA_DIR = find_data_dir()


Searching for ROGII dataset...
  V Using /home/samer/Documents/competitions/ROGII/dataset (found 773 train wells)


### 3. Configuration
Define hyperparameters, model paths, and feature columns.

In [3]:
# Cell 3: Configuration

OUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle") else os.path.abspath(os.path.join(os.getcwd(), "..", "outputs"))
os.makedirs(OUT_DIR, exist_ok=True)

CONFIG = {
    "seed": 42,
    "data_dir": DATA_DIR,
    "model_path": f"{OUT_DIR}/dense_tvt_model.keras",
    "scaler_path": f"{OUT_DIR}/dense_scaler_params.pkl",
    "hidden_layers": [48],
    "dropout": 0.2,
    "kr_rate": 1e-5,
    "epochs": 100,
    "batch_size": 1024,
    "lr": 5e-5,
    "val_ratio": 0.20,
    "max_wells": None,
    "gcn": 1.0,
    "beta_1": 0.9,
    "beta_2": 0.9,
    "amsgrad": False,
    "weight_decay": 0,
    "ema_momentum": 0.9,
}

FEATURE_COLS = ["MD", "X", "Y", "Z", "GR", "TVT_input"]
TARGET = "TVT"

print(f"Data dir   : {CONFIG['data_dir']}")
print(f"Model path : {CONFIG['model_path']}")
print(f"Features   : {FEATURE_COLS}")


Data dir   : /home/samer/Documents/competitions/ROGII/dataset
Model path : /home/samer/Documents/competitions/ROGII/outputs/dense_tvt_model.keras
Features   : ['MD', 'X', 'Y', 'Z', 'GR', 'TVT_input']


### 4. Data Helpers & Preparation
Functions for loading, preprocessing, and preparing the flat dataset for Dense training.

In [4]:
"""Data loading, preprocessing and preparation functions using Polars and JAX.

The Conv1D model is trained with a test-like TVT_input mask. In the competition test
wells, TVT_input is known before the submission interval and missing throughout the
interval that must be predicted. If validation keeps the true TVT_input, the model can
learn an identity shortcut and report unrealistically low validation RMSE.
"""

from typing import Dict, List, Tuple, Union
import glob
import os
import pickle
import jax
import jax.numpy as jnp
import numpy as np
import polars as pl

FEATURE_COLS: List[str] = ["MD", "X", "Y", "Z", "GR", "TVT_input"]
TARGET: str = "TVT"
TVT_INPUT_COL: str = "TVT_input"
MASK_START_RATIO_RANGE: Tuple[float, float] = (0.20, 0.40)


def mask_tvt_input_for_prediction_zone(
    df: pl.DataFrame,
    rng: np.random.Generator,
    ratio_range: Tuple[float, float] = MASK_START_RATIO_RANGE,
) -> Tuple[pl.DataFrame, int]:
    """Masks a suffix of TVT_input to mimic the hidden test prediction interval.

    Args:
        df (pl.DataFrame): Input DataFrame containing the well data.
        rng (np.random.Generator): Random number generator for reproducibility.
        ratio_range (Tuple[float, float]): Range of mask start ratio relative
          to well depth.

    Returns:
        Tuple[pl.DataFrame, int]: A tuple containing the masked DataFrame and
          the index where the masking started.
    """
    if TVT_INPUT_COL not in df.columns or len(df) < 2:
        return df, len(df)

    low, high = ratio_range
    mask_start = int(round(len(df) * rng.uniform(low, high)))
    mask_start = min(max(mask_start, 1), len(df) - 1)
    row_nr = pl.int_range(0, pl.len())
    df = df.with_columns(
        pl.when(row_nr >= mask_start)
        .then(None)
        .otherwise(pl.col(TVT_INPUT_COL))
        .alias(TVT_INPUT_COL)
    )
    return df, mask_start


def preprocess(df: pl.DataFrame) -> pl.DataFrame:
    """Interpolates and fills null values in the feature columns.

    Args:
        df (pl.DataFrame): Input DataFrame with missing values.

    Returns:
        pl.DataFrame: Preprocessed DataFrame with all feature columns filled.
    """
    for col in FEATURE_COLS:
        if col in df.columns:
            df = df.with_columns(
                pl.col(col)
                .interpolate()
                .fill_null(strategy="forward")
                .fill_null(strategy="backward")
                .fill_null(0.0)
            )
    return df


def prepare_data(
    data_dir: str,
    val_ratio: float = 0.20,
    seed: int = 42,
    max_wells: Union[int, None] = None,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, Dict]:
    """Loads wells and builds train/validation arrays with test-like masking.

    Reshapes the inputs to 3D tensors compatible with Conv1D layers and uses JAX
    on CPU memory for double-precision normalization to avoid GPU OOM.

    Args:
        data_dir (str): Path to the training dataset.
        val_ratio (float): Fraction of wells to use for validation.
        seed (int): Random seed for split and masking reproducibility.
        max_wells (Union[int, None]): Maximum number of wells to load (for debugging).

    Returns:
        Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, Dict]:
          A tuple containing:
            - X_train: Preprocessed standardized training features (3D: [samples, 1, features]).
            - y_train: Standardized training target values (1D: [samples]).
            - X_val: Preprocessed standardized validation features (3D: [samples, 1, features]).
            - y_val: Standardized validation target values (1D: [samples]).
            - y_val_raw: Raw validation targets in original scale (1D: [samples]).
            - scaler: Dictionary containing standardization scaling parameters.
    """
    dataset_root = os.path.dirname(data_dir)

    # 1. Get original training IDs
    pattern_train = os.path.join(data_dir, "train", "*__horizontal_well.csv")
    files_train = sorted([os.path.basename(f) for f in glob.glob(pattern_train)])

    # 2. Get augmented training files
    # pattern_aug = os.path.join(
    #     dataset_root, "dataset_augmented", "train", "*__horizontal_well_aug.csv"
    # )
    # files_aug = sorted([os.path.basename(f) for f in glob.glob(pattern_aug)])

    # Combined training file list
    files_train_all = files_train #+ files_aug

    if max_wells:
        files_train_all = files_train_all[:max_wells]

    rng = np.random.default_rng(seed)
    n_val = max(1, int(len(files_train_all) * val_ratio))
    val_set = set(rng.permutation(len(files_train_all))[:n_val])

    train_files = [fname for i, fname in enumerate(files_train_all) if i not in val_set]
    val_files = [fname for i, fname in enumerate(files_train_all) if i in val_set]

    print(
        f"Loading {len(files_train_all)} training wells total. "
        # f"(Original: {len(files_train)}, Augmented: {len(files_aug)})."
    )

    def get_arrays(files: List[str], split_name: str) -> Tuple[np.ndarray, np.ndarray]:
        """Reads CSV files for a split, masks TVT_input, and returns merged arrays.

        Args:
            files (List[str]): List of filenames to load.
            split_name (str): Label for the dataset split (e.g. "Train", "Val").

        Returns:
            Tuple[np.ndarray, np.ndarray]: Merged features and target arrays.
        """
        feats, tgts, mask_starts = [], [], []
        for filename in files:
            try:
                # if filename in files_aug:
                #     path = os.path.join(
                #         dataset_root, "dataset_augmented", "train", filename
                #     )
                # else:
                path = os.path.join(data_dir, "train", filename)

                df = pl.read_csv(path, infer_schema_length=10000)
                df = df.filter(pl.col(TARGET).is_not_null())
                if len(df) == 0:
                    continue

                df, mask_start = mask_tvt_input_for_prediction_zone(df, rng)
                df = preprocess(df)

                feats.append(df.select(FEATURE_COLS).to_numpy().astype(np.float64))
                tgts.append(df.select(TARGET).to_numpy().ravel().astype(np.float64))
                mask_starts.append(mask_start / len(df))
            except Exception as e:
                print(f"  Skip {filename}: {e}")

        if not feats:
            raise ValueError(f"No usable wells found for {split_name} split.")

        print(
            f"{split_name} TVT_input mask start ratio: "
            f"mean={np.mean(mask_starts):.3f}, "
            f"min={np.min(mask_starts):.3f}, max={np.max(mask_starts):.3f}"
        )
        return np.concatenate(feats), np.concatenate(tgts)

    X_train_raw, y_train_raw = get_arrays(train_files, "Train")
    X_val_raw, y_val_raw = get_arrays(val_files, "Val")

    cpu_dev = jax.devices("cpu")[0]
    X_tr_jax = jax.device_put(X_train_raw, cpu_dev)
    y_tr_jax = jax.device_put(y_train_raw, cpu_dev)
    X_val_jax = jax.device_put(X_val_raw, cpu_dev)
    y_val_jax = jax.device_put(y_val_raw, cpu_dev)

    # Apply log normalization to features and target
    print("Applying arcsinh transformation...")
    X_train_n = jnp.arcsinh(X_tr_jax)
    y_train_n = jnp.arcsinh(y_tr_jax)
    X_val_n = jnp.arcsinh(X_val_jax)
    y_val_n = jnp.arcsinh(y_val_jax)


    # --- DEBUG: Check for non-finite values after log transform (as float64) ---
    print(f"Train X (float64) has NaNs: {jnp.isnan(X_train_n).any()}")
    print(f"Train y (float64) has NaNs: {jnp.isnan(y_train_n).any()}")
    print(f"Train X (float64) has Infs: {jnp.isinf(X_train_n).any()}")
    print(f"Train y (float64) has Infs: {jnp.isinf(y_train_n).any()}")
    # --------------------------------------------------------------------

    scaler = {
        "feature_cols": FEATURE_COLS,
        "normalization_type": "arcsinh",
        "tvt_input_masked_for_training": True,
        "mask_start_ratio_range": MASK_START_RATIO_RANGE,
    }

    # Reshape features to 3D format (samples, sequence_length=1, features) for Conv1D compatibility
    X_train_2d = np.array(X_train_n, dtype=np.float32)
    X_val_2d = np.array(X_val_n, dtype=np.float32)

    # --- DEBUG: Check for non-finite values after casting to float32 ---
    print(f"\nTrain X (float32) has NaNs: {np.isnan(X_train_2d).any()}")
    print(f"Train y (float32) has NaNs: {np.isnan(np.array(y_train_n, dtype=np.float32)).any()}")
    print(f"Train X (float32) has Infs: {np.isinf(X_train_2d).any()}")
    print(f"Train y (float32) has Infs: {np.isinf(np.array(y_train_n, dtype=np.float32)).any()}")
    print("-----------------------------------------------------------\n")
    # --------------------------------------------------------------------

    return (
        X_train_2d,
        np.array(y_train_n, dtype=np.float32),
        X_val_2d,
        np.array(y_val_n, dtype=np.float32),
        y_val_raw,
        scaler,
    )


print("Preparing dataset...")
X_train, y_train, X_val, y_val, y_val_raw, scaler = prepare_data(DATA_DIR)
print(f"Train size: {X_train.shape}, Val size: {X_val.shape}")
print(X_train[0:10], y_train[0:10])
OUT_DIR = "../outputs"
os.makedirs(OUT_DIR, exist_ok=True)
scaler_path = os.path.join(OUT_DIR, "conv_tr_scaler_params.pkl")
with open(scaler_path, "wb") as f:
    pickle.dump(scaler, f)
print(f"Scaler parameters saved to {scaler_path}")


Preparing dataset...
Loading 773 training wells total. 
Train TVT_input mask start ratio: mean=0.301, min=0.200, max=0.400
Val TVT_input mask start ratio: mean=0.295, min=0.200, max=0.399
Applying arcsinh transformation...
Train X (float64) has NaNs: False
Train y (float64) has NaNs: False
Train X (float64) has Infs: False
Train y (float64) has Infs: False

Train X (float32) has NaNs: False
Train y (float32) has NaNs: False
Train X (float32) has Infs: False
Train y (float32) has Infs: False
-----------------------------------------------------------

Train size: (4074081, 6), Val size: (1018174, 6)
[[10.050009  15.61085   14.589781  -9.845472   5.28954   10.035097 ]
 [10.050096  15.61085   14.589781  -9.845579   5.3025584 10.035189 ]
 [10.050182  15.61085   14.589781  -9.845685   5.324032  10.035279 ]
 [10.050268  15.61085   14.589781  -9.845791   5.2783318 10.035371 ]
 [10.050355  15.61085   14.589781  -9.845897   5.2304425 10.035461 ]
 [10.050441  15.61085   14.589781  -9.846003   5.

### 5. Build & Train Dense Model
Construct the MLP architecture and execute training with callbacks.

In [5]:
# Cell 5: Build & train Dense model

def build_dense_model(input_shape, cfg):
    """Builds a simple Multi-Layer Perceptron (MLP)."""
    inp = keras.Input(shape=input_shape)
    x = inp
    for units in cfg["hidden_layers"]:
        x = layers.Dense(units, activation="leaky_relu", kernel_regularizer=regularizers.L2(cfg["kr_rate"]))(x)
        if cfg["dropout"] > 0:
            x = layers.Dropout(cfg["dropout"])(x)
    
    out = layers.Dense(1, activation="linear")(x)
    m = keras.Model(inp, out, name="Dense_TVT")

    
    m.compile(
        optimizer=keras.optimizers.Adam(
            learning_rate= cfg["lr"], 
            global_clipnorm=cfg["gcn"],
            beta_1=cfg["beta_1"],
            beta_2=cfg["beta_2"],
            amsgrad=cfg["amsgrad"],
            weight_decay=cfg["weight_decay"],
            use_ema=True,
            ema_momentum=cfg["ema_momentum"],
            ),
        loss="huber",
        metrics=[keras.metrics.RootMeanSquaredError(name="rmse")],
    )
    return m

model = build_dense_model((len(FEATURE_COLS),), CONFIG)
model.summary()

cbs = [
    callbacks.ModelCheckpoint(CONFIG["model_path"], monitor="val_rmse", save_best_only=True, mode="min"),
    callbacks.ReduceLROnPlateau(monitor="val_rmse", factor=0.5, patience=100, min_lr=3e-7),
    # callbacks.EarlyStopping(monitor="val_rmse", patience=7, restore_best_weights=True)
]

print(f"\nStarting training...")
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=CONFIG["epochs"],
    batch_size=CONFIG["batch_size"],
    callbacks=cbs
)


Model: "Dense_TVT"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 6)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 48)             │           336 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 48)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            49 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 385 (1.50 KB)

 Trainable params: 385 (1.50 KB)

 Non-trainable params: 0 (0.00 B)


Starting training...
Epoch 1/100
3979/3979 ━━━━━━━━━━━━━━━━━━━━ 9s 2ms/step - loss: 2.0409 - rmse: 4.5016 - val_loss: 0.0051 - val_rmse: 0.0995 - learning_rate: 5.0000e-05
Epoch 2/100
3979/3979 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step - loss: 0.2360 - rmse: 0.7082 - val_loss: 0.0035 - val_rmse: 0.0820 - learning_rate: 5.0000e-05
Epoch 3/100
3979/3979 ━━━━━━━━━━━━━━━━━━━━ 6s 1ms/step - loss: 0.2264 - rmse: 0.6919 - val_loss: 0.0033 - val_rmse: 0.0799 - learning_rate: 5.0000e-05
Epoch 4/100
3979/3979 ━━━━━━━━━━━━━━━━━━━━ 6s 1ms/step - loss: 0.2189 - rmse: 0.6792 - val_loss: 0.0028 - val_rmse: 0.0740 - learning_rate: 5.0000e-05
Epoch 5/100
3979/3979 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step - loss: 0.2115 - rmse: 0.6662 - val_loss: 0.0015 - val_rmse: 0.0527 - learning_rate: 5.0000e-05
Epoch 6/100
3979/3979 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step - loss: 0.2037 - rmse: 0.6528 - val_loss: 0.0024 - val_rmse: 0.0681 - learning_rate: 5.0000e-05
Epoch 7/100
3979/3979 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step - loss: 0.1963 - 

### 6. Evaluate Model
Reload the best weights and calculate final validation RMSE on raw scale.

In [6]:
# Cell 6: Evaluate (RAW SCALE)

best_model = keras.saving.load_model(CONFIG["model_path"])
yp = best_model.predict(X_val, batch_size=1024).ravel()

# Inverse transform predictions to get back to raw scale
yp_raw = np.sinh(yp)

err = yp_raw - y_val_raw
rmse = float(np.sqrt(np.mean(np.square(err))))
print(f"\nFinal Val RMSE (Dense model, RAW SCALE): {rmse:.4f}")


995/995 ━━━━━━━━━━━━━━━━━━━━ 1s 717us/step

Final Val RMSE (Dense model, RAW SCALE): 64.5447


### 6. Evaluate Model
Reload the best weights and calculate final validation RMSE on raw scale.

In [7]:
# Cell 6: Evaluate (RAW SCALE)

best_model = keras.saving.load_model(CONFIG["model_path"])
yp = best_model.predict(X_val, batch_size=1024).ravel()

# Inverse transform predictions to get back to raw scale
yp_raw = np.sinh(yp)

err = yp_raw - y_val_raw
rmse = float(np.sqrt(np.mean(np.square(err))))
print(f"\nFinal Val RMSE (Dense model, RAW SCALE): {rmse:.4f}")


995/995 ━━━━━━━━━━━━━━━━━━━━ 1s 789us/step

Final Val RMSE (Dense model, RAW SCALE): 64.5447
